In [44]:
import pandas as pd
import matplotlib.pyplot as plt

In [45]:
df = pd.read_csv('YearDataDailyCollector.csv',sep=',')
df.columns

Index(['Timestamp', 'Loud laughter', 'Cried', 'Tender moments', 'Fought',
       'Did you read a book today?',
       'What books did you finish reading? (leave blank for none)',
       'Did you watch TV today?',
       'What shows/movies did you finish reading? (leave blank for none)',
       'Michael - Emotion (5 neutral) ', 'Melanie - Emotion (5 neutral)',
       'Did dishes?', 'Miles of Cardio',
       'Did you run non-work errands on your bicycle?',
       'Did strength training?', 'Went climbing?', 'Spent time with family?',
       'Poops', 'Cups of coffee', 'Did you both consume fruit today?',
       'Did you both consume vegetables today?',
       'Travel to work (select all that apply)', 'Worst part of the day',
       'Best part of the day', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26'],
      dtype='object')

In [46]:
df['Worst part of the day'].to_csv('worst_part_of_day.csv',index=False)
df['Best part of the day'].to_csv('best_part_of_day.csv',index=False)

In [47]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.util import bigrams
from collections import Counter
import re
import string

def process_text(texts):
    """
    Process a list of texts to extract and clean 1-grams and verb-noun 2-grams.
    
    Args:
        texts (list): List of strings containing the text entries
        
    Returns:
        tuple: (Counter of 1-grams, Counter of verb-noun 2-grams)
    """
    # Download required NLTK data
    nltk.download('punkt')
    nltk.download('averaged_perceptron_tagger')
    nltk.download('wordnet')
    nltk.download('stopwords')
    
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    # Custom stop words you might want to add
    custom_stops = {'day', 'today', 'got', 'went', 'had', 'made', 'really', 'get'}
    stop_words.update(custom_stops)
     
    unigrams = []
    verb_noun_bigrams = []
    
    for text in texts:
        # Convert to lowercase and remove possessives
        text = text.lower()
        text = re.sub(r"'s\b", "", text)
        
        # Remove punctuation
        text = text.translate(str.maketrans("", "", string.punctuation))
        
        # Tokenize and tag parts of speech
        tokens = word_tokenize(text)
        # replace mel with melanie
        tokens = [re.sub(r'mel', 'melanie', token) for token in tokens]
        
        # replace feeling with feel
        tokens = [re.sub(r'feeling|felt', 'feel', token) for token in tokens]
        # replace cat, cats, kitty, kittens, kitties with cat
        cats = ['cats', 'cat', 'kitty', 'kittens', 'kitties']
        tokens = [re.sub(r'cats|cat|kitty|kittens|kitties', 'cat', token) for token in tokens]
        # combine cuddle and snuggle into one
        tokens = [re.sub(r'cuddle|snuggle', 'cuddle', token) for token in tokens]
        # combine sex and sexy
        tokens = [re.sub(r'sexy|sex', 'sex', token) for token in tokens]
        # ski and skiing
        tokens = [re.sub(r'skiing|ski', 'ski', token) for token in tokens]
        # swim and swimming
        tokens = [re.sub(r'swimming|swim', 'swim', token) for token in tokens]


        pos_tags = pos_tag(tokens)
        
        # Process unigrams
        cleaned_tokens = []
        for word, tag in pos_tags:
            # Skip stop words and short words
            if word in stop_words or len(word) < 3:
                continue
                
            # Lemmatize based on POS tag
            if tag.startswith('V'):
                lemma = lemmatizer.lemmatize(word, pos='v')
            elif tag.startswith('N'):
                lemma = lemmatizer.lemmatize(word, pos='n')
            elif tag.startswith('R'):
                lemma = lemmatizer.lemmatize(word, pos='r')
            else:
                lemma = lemmatizer.lemmatize(word)
                
            cleaned_tokens.append(lemma)
        
        unigrams.extend(cleaned_tokens)
        
        # Extract verb-noun bigrams
        for i in range(len(pos_tags)-1):
            word1, tag1 = pos_tags[i]
            word2, tag2 = pos_tags[i+1]
            
            combos = [('V','N'),('J','N')]

            for pair in combos:
                # Check for verb-noun patterns
                if tag1.startswith(pair[0]) and tag2.startswith(pair[1]):
                    # Lemmatize both words
                    word1 = lemmatizer.lemmatize(word1.lower(), pos=get_wordnet_pos(pair[0].lower()))
                    word2 = lemmatizer.lemmatize(word2.lower(), pos=get_wordnet_pos(pair[1].lower()))
                    
                    verb_noun_bigrams.append(f"{word1} {word2}")
    all_words = unigrams + verb_noun_bigrams
    return Counter(unigrams), Counter(verb_noun_bigrams), all_words

def get_wordnet_pos(treebank_tag):
    """Convert Penn Treebank POS tags to WordNet POS tags"""
    if treebank_tag.startswith('J'):
        return 'a'  # adjective
    elif treebank_tag.startswith('V'):
        return 'v'  # verb
    elif treebank_tag.startswith('N'):
        return 'n'  # noun
    elif treebank_tag.startswith('R'):
        return 'r'  # adverb
    else:
        return 'n'  # default to noun


def get_wordcloud_data(texts, min_freq=2):
    """
    Process texts and return data suitable for word cloud visualization.
    
    Args:
        texts (list): List of text entries
        min_freq (int): Minimum frequency threshold for words
        
    Returns:
        dict: Word frequencies suitable for word cloud
    """
    unigrams, bigrams, all_words = process_text(texts)
    
    # Combine unigrams and bigrams, filtering by minimum frequency
    word_freq = {
        word: freq 
        for word, freq in {**unigrams, **bigrams}.items() 
        if freq >= min_freq
    }
    
    return word_freq, all_words


In [48]:
# Process the texts
word_frequencies, all_words = get_wordcloud_data(df['Worst part of the day'])

# Print top words and their frequencies
for word, freq in sorted(word_frequencies.items(), key=lambda x: x[1], reverse=True):
    # print(f"{word}: {freq}")
    pass

# write all words to a file
with open('all_words_worst.txt', 'w') as f:
    for word in all_words:
        f.write(f"{word.replace(' ','~')}\n")

# same thing for best part of the day
word_frequencies, all_words = get_wordcloud_data(df['Best part of the day'])

# Print top words and their frequencies
for word, freq in sorted(word_frequencies.items(), key=lambda x: x[1], reverse=True):
    print(f"{word}: {freq}")

# write all words to a file
with open('all_words_best.txt', 'w') as f:
    for word in all_words:
        f.write(f"{word.replace(' ','~')}\n")

[nltk_data] Downloading package punkt to /Users/michael/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/michael/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /Users/michael/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/michael/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/michael/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/michael/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /Users/michael/nltk_data...
[nltk_data]   Package wordn

go: 81
get: 75
michael: 51
run: 46
cat: 46
melanie: 44
watch: 42
good: 41
like: 35
work: 29
see: 28
together: 28
read: 28
morning: 26
dinner: 25
cuddle: 25
talk: 21
new: 19
ski: 19
book: 18
feel: 17
take: 17
time: 17
play: 16
session: 16
little: 16
sex: 16
fun: 16
walk: 16
house: 15
client: 14
hang: 14
friend: 14
lot: 14
party: 14
bike: 13
home: 13
job: 13
people: 12
make: 12
climb: 12
cute: 12
find: 12
well: 11
part: 11
finish: 11
nice: 11
game: 10
tonight: 10
first: 10
productive: 10
thing: 10
beautiful: 10
love: 9
therapy: 9
date: 9
shop: 9
hour: 9
excite: 9
coffee: 9
apartment: 8
think: 8
try: 8
family: 8
say: 8
nap: 8
best: 8
two: 8
ever: 8
eat: 8
ride: 8
night: 7
movie: 7
listen: 7
powder: 7
surprise: 7
around: 7
amazing: 7
end: 7
right: 7
boulder: 7
pearl: 7
street: 7
way: 7
come: 7
delicious: 7
best part: 7
’ t: 7
dance: 6
life: 6
big: 6
skate: 6
realize: 6
theo: 6
back: 6
couch: 6
office: 6
mine: 6
favorite: 6
probably: 6
finally: 6
snuggly: 6
shopping: 6
buy: 6
give: 6
swim: 

In [49]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/michael/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/michael/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True